# E1 declared-source preparation receiving
Owner: uibcdf/bindingmt#78. See the dated prescription and separately retained execution receipts.
Requires the registered environment and input/output paths. Cells execute molecular work; notebook outputs remain empty.


In [ ]:
import gzip
import hashlib
import importlib.metadata as metadata
import json
import os
import platform
import sys
from copy import copy as shallow_copy
from functools import wraps
from pathlib import Path
from types import SimpleNamespace

source_dir = Path(os.environ['BINDINGMT_E1_AUTHORITY_EVIDENCE'])
output = Path(os.environ['BINDINGMT_E1_PREPARED_OUTPUT'])
output.mkdir(parents=True, exist_ok=False)
source_bytes = (source_dir / 'normal/declared_authority9.h5msm.gz').read_bytes()
assert (
    hashlib.sha256(source_bytes).hexdigest()
    == 'b518b813738fe5e9e5d72817858359d0320e321ce425fcf04af0354a10d16575'
)
source_native = gzip.decompress(source_bytes)
assert (
    hashlib.sha256(source_native).hexdigest()
    == 'fd01a684d624e8abaa19ee6d16d62ab0b84927c3abd74d5bbd5dcc3881c712b7'
)
source_path = output / 'declared_authority9.h5msm'
source_path.write_bytes(source_native)

In [ ]:
import molsysmt as msm
import numpy as np
import pyunitwizard as puw
from smonitor.integrations import CatalogException

import bindingmt as dmt

budgets = dict(
    convert=80,
    get=340,
    copy=12,
    set=16,
    permute_atoms=2,
    add_missing_hydrogens=2,
    assign_partial_charges=2,
    assign_autodock_atom_types=2,
    prepare_ligand=4,
    get_hydrogen_inventory=8,
    get_atom_correspondences=10,
    compose_atom_maps=4,
    read_poses=4,
    get_torsion_tree=8,
)
counts = dict.fromkeys(budgets, 0)
api = SimpleNamespace()


def counted(name, function):
    @wraps(function)
    def call(*args, **kwargs):
        if counts[name] >= budgets[name]:
            raise RuntimeError('Registered direct-call budget exceeded: ' + name)
        counts[name] += 1
        return function(*args, **kwargs)

    return call


for name in ['convert', 'get', 'copy', 'set', 'permute_atoms', 'compose_atom_maps']:
    setattr(api, name, counted(name, getattr(msm, name)))
for name in [
    'add_missing_hydrogens',
    'assign_partial_charges',
    'assign_autodock_atom_types',
]:
    setattr(api, name, counted(name, getattr(msm.build, name)))
api.prepare_ligand = counted('prepare_ligand', dmt.prepare_ligand)
api.get_hydrogen_inventory = counted(
    'get_hydrogen_inventory', msm.physchem.get_hydrogen_inventory
)
api.get_atom_correspondences = counted(
    'get_atom_correspondences', msm.chemical_states.get_atom_correspondences
)
api.read_poses = counted('read_poses', msm.form.string_pdbqt_text.read_poses)
api.get_torsion_tree = counted(
    'get_torsion_tree', msm.form.string_pdbqt_text.get_torsion_tree
)


def inspection_json(value):
    if puw.is_quantity(value):
        return puw.convert(value, to_form='record').to_dict()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError('Unsupported inspection value: ' + type(value).__name__)


def save(path, value):
    with path.open('x') as stream:
        json.dump(value, stream, default=inspection_json, indent=2, allow_nan=False)
        stream.write('\n')


def fingerprint(value):
    return hashlib.sha256(
        json.dumps(
            value, sort_keys=True, default=inspection_json, allow_nan=False
        ).encode()
    ).hexdigest()


def xyz(model):
    return np.asarray(
        puw.get_value(api.get(model, element='atom', coordinates=True), to_unit='nm')
    )


def charges(model):
    values = api.get(model, element='atom', partial_charge=True)
    # Native numeric storage is canonically elementary_charge (provider contract).
    if puw.is_quantity(values):
        values = puw.get_value(values, to_unit='elementary_charge')
    return np.asarray(values, dtype=float)


def chemical_payload(model):
    return api.convert(
        model.chemical_states, to_form='molsysmt.ChemicalStatesDict'
    ).to_dict()


def mechanics_inspection(model):
    mm = model.molecular_mechanics
    return dict(
        scope='Inspection only; no complete MM persistence/replay contract',
        native_values=mm.to_dict(),
        native_partial_charge_unit='elementary_charge',
        partial_charge_assignment=mm.partial_charge_assignment,
        atom_type_assignment=mm.atom_type_assignment,
    )


def model_snapshot(model):
    return dict(
        chemical=fingerprint(chemical_payload(model)),
        mechanics=fingerprint(mechanics_inspection(model)),
        coordinates_nm=xyz(model).tolist(),
        atom_ids=list(api.get(model, element='atom', atom_id=True)),
        atom_names=list(api.get(model, element='atom', name=True)),
        elements=list(api.get(model, element='atom', atom_type=True)),
        association=api.get(model, structure_chemical_state_index=True),
    )


def witness(query, reference):
    n = api.get(reference, n_atoms=True)
    candidate = np.column_stack((np.arange(n), np.arange(n)))
    report = api.get_atom_correspondences(
        query,
        reference_molecular_system=reference,
        structure_indices=[0],
        reference_structure_indices=[0],
        chemical_state='reference',
        reference_chemical_state='reference',
        candidate_atom_correspondences=candidate,
        max_matches=1,
    )
    assert report['n_candidate_matches'] == 1 and report['search_complete']
    assert report['search_scope'] == 'supplied_atom_correspondences'
    np.testing.assert_array_equal(report['atom_correspondences'], candidate[None])
    return report


def chemical_only_file(model, path):
    chemical = api.convert(
        [model.topology, model.chemical_states, model.structures],
        to_form='molsysmt.MolSys',
    )
    api.set(chemical, structure_chemical_state_index=[0])
    assert all(
        value is None for value in chemical.molecular_mechanics.to_dict().values()
    )
    assert chemical.molecular_mechanics.partial_charge_assignment is None
    assert chemical.molecular_mechanics.atom_type_assignment is None
    api.convert(chemical, to_form=path)


def refusal(label, operation, exception_type, marker):
    try:
        operation()
    except exception_type as error:
        assert marker.lower() in str(error).lower(), str(error)
        return dict(
            control=label,
            disposition='refused',
            exception_type=type(error).__name__,
            code=getattr(error, 'code', None),
            message=str(error),
        )
    raise AssertionError('Expected refusal did not occur: ' + label)

In [ ]:
profiles = [
    (
        'normal',
        [
            'nm',
            'ps',
            'kelvin',
            'mole',
            'dalton',
            'e',
            'kJ/mol',
            'kJ/(mol*nm)',
            'kJ/(mol*nm**2)',
            'radians',
        ],
    ),
    (
        'pm_fs',
        [
            'pm',
            'fs',
            'kelvin',
            'mole',
            'dalton',
            'coulomb',
            'kJ/mol',
            'kJ/(mol*pm)',
            'kJ/(mol*pm**2)',
            'radians',
        ],
    ),
]
caller_policy = puw.configure.report()
observations = []
physical_values = []
for profile, units in profiles:
    directory = output / profile
    directory.mkdir()
    with puw.context(standard_units=units):
        profile_policy = puw.configure.report()
        source = api.convert(source_path, to_form='molsysmt.MolSys')
        before9 = model_snapshot(source)
        assert api.get(source, n_atoms=True) == 9
        inventory9 = api.get_hydrogen_inventory(
            source, structure_indices=[0], chemical_state='reference'
        )
        assert inventory9['status'] == 'available'
        assert int(np.sum(inventory9['missing_hydrogen_counts'])) == 9
        assert len(inventory9['explicit_hydrogen_atom_indices']) == 0
        save(directory / 'inventory9.json', inventory9)
        expanded = api.add_missing_hydrogens(
            source,
            mode='fixed_chemical_state',
            pH=None,
            engine='RDKit',
            chemical_state='reference',
            structure_indices=[0],
            attribute_policy='strict',
            return_report=True,
        )
        model18 = expanded['molecular_system']
        h_report = expanded['report']
        save(directory / 'hydrogen_addition.json', h_report)
        assert (
            api.get(model18, n_atoms=True) == 18 and h_report['n_added_hydrogens'] == 9
        )
        assert (
            h_report['original_coordinates_preserved']
            and not h_report['dropped_attributes']
        )
        np.testing.assert_array_equal(
            xyz(model18)[:, :9], np.asarray(before9['coordinates_nm'])
        )
        for attribute, expected in [
            ('atom_id', before9['atom_ids']),
            ('name', before9['atom_names']),
            ('atom_type', before9['elements']),
        ]:
            np.testing.assert_array_equal(
                api.get(model18, element='atom', **{attribute: True})[:9], expected
            )
        assert np.asarray(h_report['parent_hydrogen_pairs']).shape == (9, 2)
        np.testing.assert_array_equal(
            np.sort(np.asarray(h_report['parent_hydrogen_pairs'])[:, 1]),
            np.arange(9, 18),
        )
        np.testing.assert_array_equal(
            h_report['atom_correspondence'],
            np.column_stack((np.arange(9), np.arange(9))),
        )
        parents = np.asarray(h_report['parent_hydrogen_pairs'])[:, 0]
        assert np.all((parents >= 0) & (parents < 9))
        inventory18 = api.get_hydrogen_inventory(
            model18, structure_indices=[0], chemical_state='reference'
        )
        assert inventory18['status'] == 'available'
        assert len(inventory18['explicit_hydrogen_atom_indices']) == 9
        assert int(np.sum(inventory18['missing_hydrogen_counts'])) == 0
        save(directory / 'inventory18.json', inventory18)
        before18 = model_snapshot(model18)
        charged = api.assign_partial_charges(
            model18,
            method='gasteiger_marsili',
            chemical_state='reference',
            structure_indices=[0],
            return_report=True,
        )
        charged18 = charged['molecular_system']
        save(directory / 'charge_assignment.json', charged['report'])
        before_charged = model_snapshot(charged18)
        typed = api.assign_autodock_atom_types(
            charged18,
            typing_scheme='autodock4',
            method='chemical_environment',
            chemical_state='reference',
            structure_indices=[0],
            return_report=True,
        )
        typed18 = typed['molecular_system']
        save(directory / 'typing_assignment.json', typed['report'])
        assert model_snapshot(charged18) == before_charged
        assert model_snapshot(model18) == before18
        before_typed = model_snapshot(typed18)
        assert fingerprint(chemical_payload(typed18)) == before18['chemical']
        np.testing.assert_array_equal(xyz(typed18), xyz(model18))
        np.testing.assert_array_equal(charges(typed18), charges(charged18))
        assert np.isfinite(charges(typed18)).all()
        assert abs(float(np.sum(charges(typed18))) - 1.0) < 1e-6
        labels18 = list(api.get(typed18, element='atom', atom_ff_type=True))
        assert (
            len(labels18) == 18
            and labels18.count('H') == 5
            and labels18.count('HD') == 4
        )
        save(
            directory / 'typed18_mechanics_inspection.json',
            mechanics_inspection(typed18),
        )
        save(
            directory / 'typed18_values.json',
            dict(
                coordinates=api.get(typed18, element='atom', coordinates=True),
                partial_charge=puw.quantity(charges(typed18), 'elementary_charge'),
                atom_ff_type=labels18,
            ),
        )
        chemical_only_file(typed18, directory / 'typed18_chemical_only.h5msm')
        physical_values.append(
            dict(
                coordinates_nm=xyz(typed18).copy(),
                charges_e=charges(typed18).copy(),
                labels=labels18,
            )
        )
        arms, prepared_models, prepared_snapshots = [], [], []
        for label, cuts, dof in [('rigid', [], 0), ('flexible', [(0, 6)], 1)]:
            prepared = api.prepare_ligand(
                typed18,
                selection='all',
                state_id='e1-new-declared-benzamidinium-20261106',
                active_torsion_bonds=cuts,
            )
            assert prepared.n_atoms == 13 and prepared.torsion_dof == dof
            assert dmt.assess_preparation(prepared)['assessment'] == 'unassessed'
            audit = dmt.audit_preparation_charges(prepared)
            assert (
                audit['assessment'] == 'consistent'
                and abs(audit['total_charge'] - 1.0) < 1e-6
            )
            native13 = prepared.to_molecular_system()
            prepared_models.append(native13)
            before13 = model_snapshot(native13)
            prepared_snapshots.append(before13)
            np.testing.assert_array_equal(charges(native13), prepared.charges)
            np.testing.assert_array_equal(
                api.get(native13, element='atom', atom_ff_type=True),
                prepared.atom_types,
            )
            assert (
                prepared.atom_types.count('HD') == 4 and 'H' not in prepared.atom_types
            )
            projection = prepared.metadata['molecular_projection']
            assert (
                projection['n_source_atoms'] == 18
                and projection['n_output_atoms'] == 13
            )
            assert len(projection['merged_source_atom_pairs']) == 5
            assert (
                abs(
                    projection['output_total_charge']
                    - projection['selected_total_charge']
                )
                <= projection['charge_conservation_tolerance']
            )
            retained = np.asarray(projection['output_to_source_atom_indices'])
            np.testing.assert_allclose(
                xyz(native13), xyz(typed18)[:, retained], rtol=0, atol=1e-12
            )
            inventory13 = api.get_hydrogen_inventory(
                native13, structure_indices=[0], chemical_state='reference'
            )
            assert inventory13['status'] == 'available'
            assert len(inventory13['explicit_hydrogen_atom_indices']) == 4
            assert int(np.sum(inventory13['missing_hydrogen_counts'])) == 5
            save(directory / f'{label}_inventory13.json', inventory13)
            save(directory / f'{label}_prepared.json', prepared.to_dict())
            save(
                directory / f'{label}_values.json',
                dict(
                    coordinates=prepared.coordinates,
                    partial_charge=puw.quantity(prepared.charges, 'elementary_charge'),
                    atom_ff_type=prepared.atom_types,
                ),
            )
            save(
                directory / f'{label}_mechanics_inspection.json',
                mechanics_inspection(native13),
            )
            save(directory / f'{label}_charge_audit.json', audit)
            chemical_path = directory / f'{label}_chemical_only.h5msm'
            chemical_only_file(native13, chemical_path)
            reread = api.convert(chemical_path, to_form='molsysmt.MolSys')
            save(
                directory / f'{label}_chemical_roundtrip_witness.json',
                witness(reread, native13),
            )
            np.testing.assert_allclose(xyz(reread), xyz(native13), rtol=0, atol=1e-12)
            assert api.get(reread, structure_chemical_state_index=True) == [0]
            order = list(range(13)) if dof == 0 else list(range(6, 13)) + list(range(6))
            assert prepared.pdbqt_atom_indices == order
            tree = dict(
                schema_version='molsysmt.pdbqt-torsion-tree@1',
                atom_ids=api.get(native13, element='atom', atom_id=True),
                fragment_atom_indices=order,
                fragment_offsets=[0, 13] if dof == 0 else [0, 7, 13],
                branch_atom_pairs=np.empty((0, 2), dtype=int)
                if dof == 0
                else np.asarray([[6, 0]]),
                branch_fragment_pairs=np.empty((0, 2), dtype=int)
                if dof == 0
                else np.asarray([[0, 1]]),
                root_fragment_index=0,
                torsdof=dof,
            )
            presentation = dict(
                atom_serials='consecutive',
                atom_names=prepared.atom_names,
                group_names=prepared.group_names,
                group_ids=[str(v) for v in prepared.group_ids],
                chain_ids='A',
                occupancy=1.0,
                b_factor=puw.quantity(0.0, 'angstrom**2'),
            )
            text, export_map = api.convert(
                native13,
                to_form='string:pdbqt_text',
                typing_scheme='autodock4',
                torsion_tree=tree,
                output_presentation=presentation,
                return_atom_map=True,
            )
            payload = text.removeprefix('pdbqt_text:')
            assert (
                export_map['payload_sha256']
                == hashlib.sha256(payload.encode()).hexdigest()
            )
            (directory / f'{label}_native.pdbqt').write_text(payload)
            save(directory / f'{label}_tree.json', tree)
            save(directory / f'{label}_export_map.json', export_map)
            np.testing.assert_array_equal(
                export_map['output_to_source_atom_indices'], order
            )
            composed = api.compose_atom_maps([h_report, projection, export_map])
            save(directory / f'{label}_source9_to_written_map.json', composed)
            assert composed['n_source_atoms'] == 9 and composed['n_output_atoms'] == 13
            assert (
                int(np.sum(np.asarray(composed['output_to_source_atom_indices']) == -1))
                == 4
            )
            dmt_payload = prepared.to_pdbqt()
            (directory / f'{label}_bindingmt.pdbqt').write_text(dmt_payload)
            native_written_tree = api.get_torsion_tree(text)
            dmt_written_tree = api.get_torsion_tree('pdbqt_text:' + dmt_payload)
            assert fingerprint(native_written_tree) == fingerprint(dmt_written_tree)
            save(directory / f'{label}_written_tree.json', native_written_tree)
            decoded_native = api.convert(
                text, to_form='molsysmt.MolSys', discard_torsion_tree=True
            )
            decoded_dmt = api.convert(
                'pdbqt_text:' + dmt_payload,
                to_form='molsysmt.MolSys',
                discard_torsion_tree=True,
            )
            for attribute in ['atom_id', 'name', 'atom_type', 'atom_ff_type']:
                np.testing.assert_array_equal(
                    api.get(decoded_native, element='atom', **{attribute: True}),
                    api.get(decoded_dmt, element='atom', **{attribute: True}),
                )
            np.testing.assert_array_equal(xyz(decoded_native), xyz(decoded_dmt))
            np.testing.assert_array_equal(charges(decoded_native), charges(decoded_dmt))
            np.testing.assert_array_equal(
                charges(decoded_native), export_map['written_partial_charge']
            )
            assert (
                abs(
                    float(np.sum(charges(decoded_native)))
                    - float(np.sum(prepared.charges))
                )
                <= export_map['charge_rounding_error_bound'] + 1e-12
            )
            np.testing.assert_allclose(
                xyz(decoded_native), xyz(native13)[:, order], rtol=0, atol=0.0000500001
            )
            np.testing.assert_allclose(
                charges(decoded_native),
                np.asarray(prepared.charges)[order],
                rtol=0,
                atol=0.000500001,
            )
            projected = api.read_poses(text, native13, export_map, return_report=True)
            pose13 = projected['molecular_system']
            save(directory / f'{label}_read_poses_report.json', projected['report'])
            save(
                directory / f'{label}_read_poses_witness.json',
                witness(pose13, native13),
            )
            np.testing.assert_allclose(
                xyz(pose13), xyz(native13), rtol=0, atol=0.0000500001
            )
            assert pose13.molecular_mechanics.partial_charge is None
            assert pose13.molecular_mechanics.atom_ff_type is None
            assert model_snapshot(native13) == before13
            arms.append(
                dict(
                    label=label,
                    n_atoms=13,
                    torsion_dof=dof,
                    written_order=order,
                    native_export_sha256=hashlib.sha256(payload.encode()).hexdigest(),
                    bindingmt_export_sha256=hashlib.sha256(
                        dmt_payload.encode()
                    ).hexdigest(),
                    chemical_h5_excludes_mm=True,
                    pose_reader_does_not_transfer_mm=True,
                    charge_audit=audit,
                    source13_checked_snapshot_unchanged=True,
                )
            )
        save(
            directory / 'rigid_flexible_witness.json',
            witness(prepared_models[0], prepared_models[1]),
        )
        altered_prepared = shallow_copy(prepared)
        altered_prepared.charges = list(prepared.charges)
        altered_prepared.charges[0] += 0.01
        altered_audit = dmt.audit_preparation_charges(altered_prepared)
        assert altered_audit['assessment'] == 'inconsistent'
        save(directory / 'edited_prepared_charge_audit.json', altered_audit)
        bad_charge, bad_type = api.copy(native13), api.copy(native13)
        api.set(bad_charge, element='atom', partial_charge=None)
        api.set(bad_type, element='atom', atom_ff_type=None)
        stale = api.permute_atoms(native13, atom_indices=list(range(12, -1, -1)))
        invalid = api.copy(native13)
        invalid_xyz = xyz(invalid).copy()
        invalid_xyz[0, 0, 0] = np.nan
        api.set(invalid, element='atom', coordinates=puw.quantity(invalid_xyz, 'nm'))

        def write_bad(model):
            return api.convert(
                model,
                to_form='string:pdbqt_text',
                typing_scheme='autodock4',
                torsion_tree=tree,
                output_presentation=presentation,
            )

        controls = [
            refusal(
                'missing_charge',
                lambda: write_bad(bad_charge),
                CatalogException,
                'charge',
            ),
            refusal(
                'missing_type', lambda: write_bad(bad_type), CatalogException, 'type'
            ),
            refusal(
                'stale_tree_after_axis_permutation',
                lambda: write_bad(stale),
                CatalogException,
                'atom_ids',
            ),
            refusal(
                'nonfinite_geometry',
                lambda: write_bad(invalid),
                CatalogException,
                'finite',
            ),
        ]
        for model, snapshot in zip(prepared_models, prepared_snapshots):
            assert model_snapshot(model) == snapshot
        assert model_snapshot(source) == before9
        assert model_snapshot(model18) == before18
        assert model_snapshot(charged18) == before_charged
        assert model_snapshot(typed18) == before_typed
        assert puw.configure.report() == profile_policy
        observation = dict(
            profile=profile,
            new_source_atoms=9,
            expanded_atoms=18,
            retained_atoms=13,
            generated_local_hydrogens=9,
            merged_nonpolar_hydrogens=5,
            retained_polar_hydrogens=4,
            prepared_arms=arms,
            controls=controls,
            source9_expanded18_charged18_typed18_checked_snapshots_unchanged=True,
            unit_policy_preserved=True,
        )
        save(directory / 'observations.json', observation)
        observations.append(observation)
    assert puw.configure.report() == caller_policy

In [ ]:
assert len(observations) == 2 and counts['prepare_ligand'] == 4
assert counts['get_atom_correspondences'] == 10
assert (
    counts['add_missing_hydrogens']
    == counts['assign_partial_charges']
    == counts['assign_autodock_atom_types']
    == 2
)
for arm in ['rigid', 'flexible']:
    for suffix in ['native.pdbqt', 'bindingmt.pdbqt']:
        assert (output / 'normal' / f'{arm}_{suffix}').read_bytes() == (
            output / 'pm_fs' / f'{arm}_{suffix}'
        ).read_bytes()
np.testing.assert_allclose(
    physical_values[0]['coordinates_nm'],
    physical_values[1]['coordinates_nm'],
    rtol=0,
    atol=1e-12,
)
np.testing.assert_allclose(
    physical_values[0]['charges_e'], physical_values[1]['charges_e'], rtol=0, atol=1e-12
)
assert physical_values[0]['labels'] == physical_values[1]['labels']
assert (source_dir / 'normal/declared_authority9.h5msm.gz').read_bytes() == source_bytes
assert not any(name == 'vina' or name.startswith('vina.') for name in sys.modules)
loaded_native = []
package_root = Path(msm.__file__).resolve().parent
for name, module in list(sys.modules.items()):
    filename = getattr(module, '__file__', None)
    if name.startswith('molsysmt.') and filename and filename.endswith('.so'):
        path = Path(filename).resolve()
        loaded_native.append(
            dict(
                module=name,
                file=path.name,
                sha256=hashlib.sha256(path.read_bytes()).hexdigest(),
                inside_selected_provider=path.is_relative_to(package_root),
                build_source_correspondence='unassessed',
            )
        )
receipt = dict(
    schema='bindingmt.e1_prepared_authority_receiving@1',
    owner='uibcdf/bindingmt#78',
    scope='One newly declared ligand, fixed-state H generation, named mechanics, rigid/flexible projections',
    python=platform.python_version(),
    os=platform.system(),
    architecture=platform.machine(),
    versions={
        name: metadata.version(name)
        for name in [
            'bindingmt',
            'molsysmt',
            'argdigest',
            'pyunitwizard',
            'numpy',
            'rdkit',
            'h5py',
        ]
    },
    loaded_provider_native_modules=loaded_native,
    profiles=observations,
    completed_profiles=len(observations),
    monitored_direct_consumer_calls=counts,
    monitored_direct_consumer_budgets=budgets,
    positive_chemical_witnesses=10,
    export_refusals=8,
    inconsistent_charge_controls=2,
    new_declared_states=1,
    unit_policy_repetitions=2,
    new_native_scoring_calls=0,
    historical_prepared13_identity_admitted=False,
    complete_mm_persistence_qualified=False,
    score_comparison_admitted=False,
    e1_accepted=False,
    claim_limit='New preparation only; H are local geometry, inspection is not a complete MM codec, no historical H correspondence or energy comparison.',
    excluded_operations=[
        'Vina',
        'search',
        'scoring',
        'optimization',
        'historical H matching',
        'interaction detection',
        'viewer',
    ],
)
save(output / 'receipt.json', receipt)
print(
    json.dumps(
        dict(
            prepared_arms=4, positive_witnesses=10, expected_refusals=8, energy_calls=0
        )
    )
)